# Train path classification model

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("is cuda available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import os

data_dir = os.path.abspath('../data/FIVES')
train_split = 'train_clean'

## Overview of the training pipeline and it's main modules

![train_pipeline](../images/train_pipeline.png "Train pipeline diagram")

## Features Generator / Features extractor
The features generator is here a pretrained UNet (see [U-Net pretraining notebook (2)](./02_pretrain_unet.ipynb))

We load the checkpoint of the pretrained UNet and use it as a features generator for our path classification model.

We replace the last 32 to 1 layers convolutional layer of the pretrained UNet by a new one with 32 output channels, and we keep the pretrained weights for the rest of the UNet. We also set `freeze_pretrained` to False to allow fine-tuning of the pretrained UNet during the training of the path classification model.

In [ ]:
from path_neural_networks.models.features_generators import FeaturesGenerator, PretrainedUnetFeaturesGenerator
from utils.other import pretty_dict_print

unet_checkpoint_dir = os.path.abspath('../checkpoints/unet_pretraining')
unet_ckpt_path = os.path.join(unet_checkpoint_dir, os.listdir(unet_checkpoint_dir)[0])
print("Using UNet checkpoint:", unet_ckpt_path)

features_generator_out_channels = 32

features_generator: FeaturesGenerator = PretrainedUnetFeaturesGenerator(
    ckpt_path=unet_ckpt_path,
    device=device,
    out_channels=features_generator_out_channels,
    freeze_pretrained=False,
    skip_connection=False
)
print("Features generator configuration:")
pretty_dict_print(features_generator.as_dict())

## Path Sampler
\#TODO

![path_sampler](../images/path_features_sampling.png)

In [ ]:
from path_neural_networks.models.path_samplers import PathSampler, MultiScaleSquarePathSampling, SamplingMaxAggregation

sampling_square_sizes = [1, 3, 5]
sampling_aggregation_method = SamplingMaxAggregation()

path_sampler: PathSampler = MultiScaleSquarePathSampling(
    in_channels=features_generator_out_channels,
    square_sizes=sampling_square_sizes,
    aggregation=sampling_aggregation_method
)
print("Path sampler configuration:")
pretty_dict_print(path_sampler.as_dict())

In [ ]:
from path_neural_networks.models.path_encoders import PathEncoder, ConvMaxPoolingPathEncoder

conv_path_residual_blocks = False
conv_path_skip_connections = False
conv_path_layers = [None, None, 256]

path_encoder: PathEncoder = ConvMaxPoolingPathEncoder(
    in_channels=path_sampler.out_channels, 
    hidden_layers=conv_path_layers, 
    skip_connection=conv_path_skip_connections, 
    residual_blocks=conv_path_residual_blocks
)
print("Path encoder configuration:")
pretty_dict_print(path_encoder.as_dict())

In [ ]:
from path_neural_networks.models.path_classifiers import PathClassifier, FCNPathClassifier

path_classifier_n_hidden_layers = 2
path_classifier_dropout = 0

path_classifier: PathClassifier = FCNPathClassifier(
    in_channels=path_encoder.out_channels,
    n_hidden_layers=path_classifier_n_hidden_layers,
    num_classes=1,
    dropout=path_classifier_dropout
)
print("Path classifier configuration:")
pretty_dict_print(path_classifier.as_dict())